In [17]:
import pandas as pd

df = pd.read_csv(
    "../dataset/final_dataset.csv"
)

df = df[["title", "label"]]

df.head()

,title,label
0,Did Miley Cyrus and Liam Hemsworth secretly ge...,0
1,Paris Jackson & Cara Delevingne Enjoy Night Ou...,0
2,Celebrities Join Tax March in Protest of Donal...,0
3,Cindy Crawford's daughter Kaia Gerber wears a ...,0
4,Full List of 2018 Oscar Nominations – Variety,0


In [18]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["title"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [19]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [20]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=64
)

In [21]:
import torch

class NewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

In [22]:
train_dataset = NewsDataset(
    train_encodings,
    train_labels
)

test_dataset = NewsDataset(
    test_encodings,
    test_labels
)

In [23]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True
)

In [25]:
import numpy as np
from transformers import Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    acc = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": acc
    }
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [26]:
trainer.train()

C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.433260,0.454514,0.851509
2,0.325283,0.558689,0.859267
3,0.341534,0.697593,0.851509


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=13917, training_loss=0.38724707950675097, metrics={'train_runtime': 10445.9406, 'train_samples_per_second': 5.329, 'train_steps_per_second': 1.332, 'total_flos': 921774393547776.0, 'train_loss': 0.38724707950675097, 'epoch': 3.0})

In [31]:
trainer.state

TrainerState(epoch=3.0, global_step=13917, max_steps=13917, logging_steps=100, eval_steps=500, save_steps=500, train_batch_size=4, num_train_epochs=3, num_input_tokens_seen=0, total_flos=921774393547776.0, log_history=[{'loss': 0.649427261352539, 'grad_norm': 2.456874370574951, 'learning_rate': 4.9644319896529426e-05, 'epoch': 0.02155636990730761, 'step': 100}, {'loss': 0.5689318466186524, 'grad_norm': 4.178120136260986, 'learning_rate': 4.9285047064740966e-05, 'epoch': 0.04311273981461522, 'step': 200}, {'loss': 0.5436323547363281, 'grad_norm': 6.797926902770996, 'learning_rate': 4.8925774232952506e-05, 'epoch': 0.06466910972192283, 'step': 300}, {'loss': 0.550029411315918, 'grad_norm': 5.030547142028809, 'learning_rate': 4.8566501401164046e-05, 'epoch': 0.08622547962923044, 'step': 400}, {'loss': 0.5488957977294922, 'grad_norm': 13.459747314453125, 'learning_rate': 4.8207228569375586e-05, 'epoch': 0.10778184953653805, 'step': 500}, {'loss': 0.5470691299438477, 'grad_norm': 10.6335906

In [35]:
trainer.save_model(
    "../models/distilbert_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:
tokenizer.save_pretrained(
    "../models/distilbert_model"
)

('../models/distilbert_model\\tokenizer_config.json',
 '../models/distilbert_model\\tokenizer.json')

In [32]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(
    predictions.predictions,
    axis=1
)

accuracy = accuracy_score(
    test_labels,
    preds
)

print("DistilBERT Accuracy:", accuracy)

C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


DistilBERT Accuracy: 0.8515086206896552


In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [34]:
trainer.evaluate()

C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


RuntimeError: on_train_begin must be called before on_evaluate